# 用训练得到的模型预测-视频

同济子豪兄 2023-7-13

## 进入mmsegmentation目录

In [1]:
# import os
# os.chdir('mmsegmentation')

import os
# 假设 mmsegmentation 的绝对路径是 /project/mmsegmentation
mmsegmentation_path = "E:/bishe_demo/mmsegmentation-main"
# 切换到 mmsegmentation 文件夹
os.chdir(mmsegmentation_path)
# 验证当前工作目录
print("当前工作目录:", os.getcwd())  # 输出: /project/mmsegmentation

当前工作目录: E:\bishe_demo\mmsegmentation-main


## MMSegmentation官方视频/摄像头预测

In [2]:
# !python3 demo/video_demo.py \
#          data/video_watermelon_3.mov \
#          Zihao-Configs/ZihaoDataset_KNet_20230712.py \
#          checkpoint/Zihao_KNet.pth \
#          --device cpu \
#          --opacity 0.5 \
#          --show

## OpenCV

### 导入工具包

In [2]:
import time
import numpy as np
from tqdm import tqdm
import cv2

import mmcv
from mmseg.apis import init_model, inference_model

### 载入训练好的模型

In [3]:
# 模型 config 配置文件
config_file = 'Zihao-Configs/ZihaoDataset_PSPNet_20230818.py'

# 模型 checkpoint 权重文件
checkpoint_file = 'work_dirs/ZihaoDataset-PSPNet/best_mIoU_iter_1000.pth'

# device = 'cpu'
device = 'cuda:0'

In [4]:
model = init_model(config_file, checkpoint_file, device=device)

E:\bishe_demo\mmsegmentation-main\mmseg\models\losses\cross_entropy_loss.py:250: UserWarning: Default ``avg_non_ignore`` is False, if you would like to ignore the certain label and average loss over non-ignore labels, which is the same with PyTorch official cross_entropy, set ``avg_non_ignore=True``.
  warnings.warn(


Loads checkpoint by local backend from path: work_dirs/ZihaoDataset-PSPNet/best_mIoU_iter_1000.pth


### 各类别的配色方案（BGR）

In [5]:
# palette = [
#     ['background', [127,127,127]],
#     ['red', [0,0,200]],
#     ['green', [0,200,0]],
#     ['white', [144,238,144]],
#     ['seed-black', [30,30,30]],
#     ['seed-white', [8,189,251]]
# ]

# 自定义 Cityscapes 19 个类别的配色方案（BGR 格式）
palette = [
    ['road', [100, 100, 100]],  # 道路
    ['sidewalk', [200, 50, 200]],  # 人行道
    ['building', [50, 50, 50]],  # 建筑
    ['wall', [150, 150, 200]],  # 墙
    ['fence', [200, 150, 150]],  # 栅栏
    ['pole', [150, 150, 150]],  # 杆子
    ['traffic light', [255, 200, 0]],  # 交通灯
    ['traffic sign', [255, 255, 0]],  # 交通标志
    ['vegetation', [0, 200, 0]],  # 植被
    ['terrain', [200, 255, 200]],  # 地形
    ['sky', [0, 150, 255]],  # 天空
    ['person', [255, 0, 0]],  # 人
    ['rider', [255, 100, 0]],  # 骑行者
    ['car', [0, 0, 255]],  # 汽车
    ['truck', [0, 0, 150]],  # 卡车
    ['bus', [0, 100, 150]],  # 公交车
    ['train', [0, 150, 150]],  # 火车
    ['motorcycle', [0, 0, 200]],  # 摩托车
    ['bicycle', [200, 0, 100]]  # 自行车
]

In [6]:
palette_dict = {}
for idx, each in enumerate(palette):
    palette_dict[idx] = each[1]

In [7]:
palette_dict

{0: [100, 100, 100],
 1: [200, 50, 200],
 2: [50, 50, 50],
 3: [150, 150, 200],
 4: [200, 150, 150],
 5: [150, 150, 150],
 6: [255, 200, 0],
 7: [255, 255, 0],
 8: [0, 200, 0],
 9: [200, 255, 200],
 10: [0, 150, 255],
 11: [255, 0, 0],
 12: [255, 100, 0],
 13: [0, 0, 255],
 14: [0, 0, 150],
 15: [0, 100, 150],
 16: [0, 150, 150],
 17: [0, 0, 200],
 18: [200, 0, 100]}

### 逐帧处理函数

In [8]:
opacity = 0.3 # 透明度，越大越接近原图

In [9]:
def process_frame(img_bgr):
    
    # 记录该帧开始处理的时间
    start_time = time.time()
    
    # 语义分割预测
    result = inference_model(model, img_bgr)
    pred_mask = result.pred_sem_seg.data[0].cpu().numpy()
    
    # 将预测的整数ID，映射为对应类别的颜色
    pred_mask_bgr = np.zeros((pred_mask.shape[0], pred_mask.shape[1], 3))
    for idx in palette_dict.keys():
        pred_mask_bgr[np.where(pred_mask==idx)] = palette_dict[idx]
    pred_mask_bgr = pred_mask_bgr.astype('uint8')
    
    # 将语义分割预测图和原图叠加显示
    pred_viz = cv2.addWeighted(img_bgr, opacity, pred_mask_bgr, 1-opacity, 0)

    return pred_viz

### 视频逐帧处理

In [10]:
# 视频逐帧处理代码模板
# 不需修改任何代码，只需定义process_frame函数即可
# 同济子豪兄 2021-7-10

def generate_video(input_path='videos/robot.mp4'):
    filehead = input_path.split('/')[-1]
    output_path = "out-" + filehead
    
    print('视频开始处理',input_path)
    
    # 获取视频总帧数
    cap = cv2.VideoCapture(input_path)
    frame_count = 0
    while(cap.isOpened()):
        success, frame = cap.read()
        frame_count += 1
        if not success:
            break
    cap.release()
    print('视频总帧数为',frame_count)
    
    # cv2.namedWindow('Crack Detection and Measurement Video Processing')
    cap = cv2.VideoCapture(input_path)
    frame_size = (cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # fourcc = int(cap.get(cv2.CAP_PROP_FOURCC))
    # fourcc = cv2.VideoWriter_fourcc(*'XVID')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = cap.get(cv2.CAP_PROP_FPS)

    out = cv2.VideoWriter(output_path, fourcc, fps, (int(frame_size[0]), int(frame_size[1])))
    
    # 进度条绑定视频总帧数
    with tqdm(total=frame_count-1) as pbar:
        try:
            while(cap.isOpened()):
                success, frame = cap.read()
                if not success:
                    break

                # 处理帧
                # frame_path = './temp_frame.png'
                # cv2.imwrite(frame_path, frame)
                try:
                    frame = process_frame(frame)
                except:
                    print('报错！', error)
                    pass
                
                if success == True:
                    # cv2.imshow('Video Processing', frame)
                    out.write(frame)

                    # 进度条更新一帧
                    pbar.update(1)

                # if cv2.waitKey(1) & 0xFF == ord('q'):
                    # break
        except:
            print('中途中断')
            pass

    cv2.destroyAllWindows()
    out.release()
    cap.release()
    print('视频已保存', output_path)

In [11]:
generate_video(input_path='data/street_20220330_174028.mp4')
#视频直接放在mmsegmentation-main/ 下面

视频开始处理 data/street_20220330_174028.mp4
视频总帧数为 139


100%|██████████| 138/138 [00:41<00:00,  3.30it/s]

视频已保存 out-street_20220330_174028.mp4
